# Implementation of a Basic Neural Network and Custom Modification

This notebook demonstrates a multi-layer perceptron (MLP) built from scratch using `NumPy` with full manual backpropagation. It includes:
1. **Part 1**: The original network using a Sigmoid activation function.
2. **Part 2**: A customized network variation utilizing the ReLU activation function.
3. **Analysis**: A conceptual breakdown of the mathematical differences.

## Part 1: Original Neural Network (Sigmoid Activation)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Seed for reproducibility
np.random.seed(42)

# 1. Helper Functions
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

# Generate toy dataset: 100 samples, 2 input features, 1 target output
X = np.random.randn(100, 2)
y = (X[:, 0] + X[:, 1] > 0).astype(float).reshape(-1, 1)  # Binary target

# 2. Network Initialization (2 Input -> 3 Hidden -> 1 Output)
input_dim = 2
hidden_dim = 3
output_dim = 1

W1 = np.random.randn(input_dim, hidden_dim) * 0.1
b1 = np.zeros((1, hidden_dim))
W2 = np.random.randn(hidden_dim, output_dim) * 0.1
b2 = np.zeros((1, output_dim))

learning_rate = 0.1
epochs = 500
sigmoid_loss_history = []

# 3. Gradient Descent Loop
for epoch in range(epochs):
    # --- Forward Pass ---
    z1 = np.dot(X, W1) + b1
    a1 = sigmoid(z1)
    z2 = np.dot(a1, W2) + b2
    a2 = sigmoid(z2)  # Final prediction
    
    # --- Compute Mean Squared Error Loss ---
    loss = np.mean(0.5 * (a2 - y) ** 2)
    sigmoid_loss_history.append(loss)
    
    # --- Backward Pass (Manual Gradients) ---
    m = X.shape[0]
    
    # Output layer error gradients
    da2 = (a2 - y) / m
    dz2 = da2 * sigmoid_derivative(z2)
    dW2 = np.dot(a1.T, dz2)
    db2 = np.sum(dz2, axis=0, keepdims=True)
    
    # Hidden layer error gradients
    da1 = np.dot(dz2, W2.T)
    dz1 = da1 * sigmoid_derivative(z1)
    dW1 = np.dot(X.T, dz1)
    db1 = np.sum(dz1, axis=0, keepdims=True)
    
    # --- Update Weights ---
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

print(f"Sigmoid Network Final Loss: {sigmoid_loss_history[-1]:.6f}")

## Part 2: Custom Tweak (ReLU Activation Network)

In [ ]:
# 1. Helper Functions for Variation
def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

# Re-initialize identical weights for a fair comparison
np.random.seed(42)
W1_relu = np.random.randn(input_dim, hidden_dim) * 0.1
b1_relu = np.zeros((1, hidden_dim))
W2_relu = np.random.randn(hidden_dim, output_dim) * 0.1
b2_relu = np.zeros((1, output_dim))

relu_loss_history = []

# 2. Gradient Descent Loop with ReLU
for epoch in range(epochs):
    # --- Forward Pass ---
    z1_r = np.dot(X, W1_relu) + b1_relu
    a1_r = relu(z1_r)            # Tweak: Hidden layer uses ReLU
    z2_r = np.dot(a1_r, W2_relu) + b2_relu
    a2_r = sigmoid(z2_r)         # Output layer remains sigmoid for binary prediction boundary
    
    # --- Compute Loss ---
    loss_r = np.mean(0.5 * (a2_r - y) ** 2)
    relu_loss_history.append(loss_r)
    
    # --- Backward Pass ---
    m = X.shape[0]
    
    # Output layer gradients
    da2_r = (a2_r - y) / m
    dz2_r = da2_r * sigmoid_derivative(z2_r)
    dW2_r = np.dot(a1_r.T, dz2_r)
    db2_r = np.sum(dz2_r, axis=0, keepdims=True)
    
    # Hidden layer gradients with modified activation derivative
    da1_r = np.dot(dz2_r, W2_relu.T)
    dz1_r = da1_r * relu_derivative(z1_r)  # Tweak: Gradient multiplied by ReLU derivative
    dW1_r = np.dot(X.T, dz1_r)
    db1_r = np.sum(dz1_r, axis=0, keepdims=True)
    
    # --- Update Weights ---
    W2_relu -= learning_rate * dW2_r
    b2_relu -= learning_rate * db2_r
    W1_relu -= learning_rate * dW1_r
    b1_relu -= learning_rate * db1_r

print(f"ReLU Network Final Loss: {relu_loss_history[-1]:.6f}")

# 3. Plot Comparison Curve
plt.figure(figsize=(8, 5))
plt.plot(sigmoid_loss_history, label='Original (Sigmoid)', color='steelblue')
plt.plot(relu_loss_history, label='Modified (ReLU)', color='coral', linestyle='--')
plt.title('Training Loss Over Epochs: Sigmoid vs. ReLU')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(True)
plt.show()

## Part 3: Written Analysis

1. **Modification Chosen**: I chose to replace the Sigmoid activation function in the hidden layer with a Rectified Linear Unit (ReLU) function.
2. **Mathematical Change**: Mathematically, the forward mapping changes from $\sigma(z) = \frac{1}{1 + e^{-z}}$ to $f(z) = \max(0, z)$. Consequently, during backpropagation, its derivative changes from a vanishing bell curve peaking at $0.25$ to a constant binary gate: $1$ for all positive inputs and $0$ for negative inputs.
3. **Behavioral Expectation**: I expected the ReLU network to optimize significantly faster and reach a lower terminal loss because its constant gradient of $1.0$ completely mitigates the vanishing gradient problem in the hidden layer, preventing early saturation during parameter update adjustments.